In [7]:
import json
import numpy as np
import pandas as pd
import warnings
import joblib
import prince

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)

warnings.filterwarnings("ignore")

In [8]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATH  = r"D:\2026\MockProject_062026_NhomAI\data\dataset_5core_template.json"        
TARGET_COL = "Care_Level"            
CORES_KEY  = "5_cores"          
TEST_SIZE  = 0.2                
VAL_SIZE   = 0.15              
MODEL_SAVE_DIR = r"D:\2026\MockProject_062026_NhomAI\model"  


In [9]:
print("=" * 60)
print("  LOAD & PREPROCESS DATA")
print("=" * 60)

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# Tự động đọc các core và feature từ toàn bộ dữ liệu
CORE_FEATURES = {}

for record in raw_data:
    cores = record.get("5_cores", record.get("5 cores", {}))

    for core_name, core_data in cores.items():
        if core_name not in CORE_FEATURES:
            CORE_FEATURES[core_name] = list(core_data.keys())

X_raw = {core_name: [] for core_name in CORE_FEATURES}
y_raw = []

for record in raw_data:
    cores = record.get("5_cores", record.get("5 cores", {}))
    for core_name, feat_list in CORE_FEATURES.items():
        core_data = cores.get(core_name, {})
        row = []
        for feat in feat_list:
            val = core_data.get(feat, 0)
            row.append(val)
        X_raw[core_name].append(row)
    y_raw.append(record.get(TARGET_COL))

print(f"Total records loaded: {len(raw_data):,}")

# ---- Encode Target ----
y = np.array(y_raw, dtype=np.int64)
num_classes = len(np.unique(y))

classes, counts = np.unique(y, return_counts=True)

print(f"Target classes: {classes}")
print(f"Class distribution: {dict(zip(classes, counts))}")

# ---- Train / Test / Val Split (Stratified) ----
indices = np.arange(len(y))

# 1. Tách tập test
train_val_idx, test_idx = train_test_split(
    indices, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# 2. Tách tập train và validation từ tập còn lại
val_ratio = VAL_SIZE / (1 - TEST_SIZE)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=val_ratio, random_state=RANDOM_STATE, stratify=y[train_val_idx]
)

y_train = y[train_idx]
y_val   = y[val_idx]
y_test  = y[test_idx]

  LOAD & PREPROCESS DATA
Total records loaded: 9,989
Target classes: [0 1 2]
Class distribution: {np.int64(0): np.int64(5320), np.int64(1): np.int64(3373), np.int64(2): np.int64(1296)}


In [10]:
print("\n" + "=" * 60)
print("  APPLYING PCA DIMENSIONALITY REDUCTION")
print("=" * 60)

famd_transformers = {}
scalers = {}

X_train_famd_list = []
X_val_famd_list   = []
X_test_famd_list  = []

for core_name, feat_list in CORE_FEATURES.items():
    # Tạo DataFrame cho core hiện tại
    df_core = pd.DataFrame(
        X_raw[core_name],
        columns=feat_list,
        dtype="float64"
    )

    # Chia train / validation / test trước khi chuẩn hóa
    df_train = df_core.iloc[train_idx].reset_index(drop=True)
    df_val   = df_core.iloc[val_idx].reset_index(drop=True)
    df_test  = df_core.iloc[test_idx].reset_index(drop=True)

    # Chuẩn hóa: chỉ fit trên tập train
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(df_train)
    X_val_scaled   = scaler.transform(df_val)
    X_test_scaled  = scaler.transform(df_test)

    scalers[core_name] = scaler

    # Chuyển lại thành DataFrame cho prince.PCA
    X_train_scaled = pd.DataFrame(
        X_train_scaled,
        columns=feat_list
    )

    X_val_scaled = pd.DataFrame(
        X_val_scaled,
        columns=feat_list
    )

    X_test_scaled = pd.DataFrame(
        X_test_scaled,
        columns=feat_list
    )

    # Giảm mỗi core xuống 1 thành phần
    pca = prince.PCA(
        n_components=1,
        n_iter=10,
        copy=True,
        check_input=True,
        random_state=RANDOM_STATE
    )

    pca.fit(X_train_scaled)

    # Giữ tên famd_transformers để không phải sửa code phía sau
    famd_transformers[core_name] = pca

    explained_variance = pca.percentage_of_variance_

    print(
        f"  Core {core_name:<35} (PCA) "
        f"-> Explained Variance (1st Component): "
        f"{explained_variance[0]:.2f}%"
    )

    X_train_famd_list.append(
        pca.transform(X_train_scaled).values.astype(np.float32)
    )

    X_val_famd_list.append(
        pca.transform(X_val_scaled).values.astype(np.float32)
    )

    X_test_famd_list.append(
        pca.transform(X_test_scaled).values.astype(np.float32)
    )


X_train_famd = np.hstack(X_train_famd_list)
X_val_famd   = np.hstack(X_val_famd_list)
X_test_famd  = np.hstack(X_test_famd_list)

print(f"\nFinal concatenated PCA Train shape: {X_train_famd.shape}")
print(f"Final concatenated PCA Val shape  : {X_val_famd.shape}")
print(f"Final concatenated PCA Test shape : {X_test_famd.shape}")


  APPLYING PCA DIMENSIONALITY REDUCTION
  Core ADLs & IADLs                        (PCA) -> Explained Variance (1st Component): 35.91%
  Core Cognitive & Neurological Status     (PCA) -> Explained Variance (1st Component): 44.33%
  Core Clinical Risk Assessments           (PCA) -> Explained Variance (1st Component): 21.97%
  Core Mood & Behavioral Health            (PCA) -> Explained Variance (1st Component): 50.87%
  Core Financial & Legal                   (PCA) -> Explained Variance (1st Component): 25.50%

Final concatenated PCA Train shape: (6492, 5)
Final concatenated PCA Val shape  : (1499, 5)
Final concatenated PCA Test shape : (1998, 5)


In [11]:
print("\n" + "=" * 60)
print("  TRAINING & COMPARING ML MODELS")
print("=" * 60)

models_to_compare = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=5, random_state=RANDOM_STATE
    ),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=6,
        random_state=RANDOM_STATE, n_jobs=-1, eval_metric="mlogloss", verbosity=0
    ),
    "LightGBM": lgb.LGBMClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=6,
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
    )
}

comparison_results = []

for name, model in models_to_compare.items():
    model.fit(X_train_famd, y_train)
    
    y_val_pred  = model.predict(X_val_famd)
    y_test_pred = model.predict(X_test_famd)
    
    val_acc     = accuracy_score(y_val, y_val_pred)
    test_acc    = accuracy_score(y_test, y_test_pred)
    weighted_f1 = f1_score(y_test, y_test_pred, average="weighted", zero_division=0)
    macro_f1    = f1_score(y_test, y_test_pred, average="macro", zero_division=0)
    
    comparison_results.append({
        "model_name": name,
        "model_object": model,
        "predictions": y_test_pred,
        "val_accuracy": val_acc,
        "test_accuracy": test_acc,
        "weighted_f1": weighted_f1,
        "macro_f1": macro_f1
    })
    print(f"  Model: {name:<20} | Val Acc: {val_acc:.4f} | Test Acc: {test_acc:.4f} | Macro F1: {macro_f1:.4f}")



  TRAINING & COMPARING ML MODELS
  Model: Random Forest        | Val Acc: 0.9199 | Test Acc: 0.9159 | Macro F1: 0.8976
  Model: Gradient Boosting    | Val Acc: 0.9173 | Test Acc: 0.9259 | Macro F1: 0.9072
  Model: XGBoost              | Val Acc: 0.9153 | Test Acc: 0.9164 | Macro F1: 0.8940
  Model: LightGBM             | Val Acc: 0.9086 | Test Acc: 0.9119 | Macro F1: 0.8905


In [12]:
print("\n" + "=" * 75)
print("  MODELS COMPARISON SUMMARY")
print("=" * 75)

comparison_df = pd.DataFrame(comparison_results)[[
    "model_name", "val_accuracy", "test_accuracy", "weighted_f1", "macro_f1"
]].sort_values("macro_f1", ascending=False)

print(comparison_df.to_string(index=False))

best_idx = next(i for i, r in enumerate(comparison_results) if r["model_name"] == comparison_df.iloc[0]["model_name"])
best_info = comparison_results[best_idx]
best_name = best_info["model_name"]

all_preds   = best_info["predictions"]
all_targets = y_test

print(f"\n=== Best Model: {best_name} ===")
print(f"   Validation Accuracy: {best_info['val_accuracy']:.4f}")
print(f"   Test Accuracy      : {best_info['test_accuracy']:.4f}")
print(f"   Weighted F1-Score  : {best_info['weighted_f1']:.4f}")
print(f"   Macro F1-Score     : {best_info['macro_f1']:.4f}")

print(f"\nDetailed Classification Report — {best_name}:")
print(classification_report(all_targets, all_preds, target_names=[str(c) for c in le_target.classes_]))


print("\nConfusion Matrix:")
print(confusion_matrix(all_targets, all_preds))



  MODELS COMPARISON SUMMARY
       model_name  val_accuracy  test_accuracy  weighted_f1  macro_f1
Gradient Boosting      0.917278       0.925926     0.926606  0.907153
    Random Forest      0.919947       0.915916     0.916499  0.897628
          XGBoost      0.915277       0.916416     0.917131  0.893980
         LightGBM      0.908606       0.911912     0.912489  0.890501

=== Best Model: Gradient Boosting ===
   Validation Accuracy: 0.9173
   Test Accuracy      : 0.9259
   Weighted F1-Score  : 0.9266
   Macro F1-Score     : 0.9072

Detailed Classification Report — Gradient Boosting:
              precision    recall  f1-score   support

         0.0       0.98      0.95      0.96      1064
         1.0       0.89      0.90      0.89       675
         2.0       0.83      0.90      0.87       259

    accuracy                           0.93      1998
   macro avg       0.90      0.92      0.91      1998
weighted avg       0.93      0.93      0.93      1998


Confusion Matrix:
[[101

In [13]:
print("\n" + "=" * 60)
print("  SAVING ALL TRAINED FAMD PIPELINES")
print("=" * 60)

import os

for r in comparison_results:
    m_name = r["model_name"]
    file_name = f"{m_name.lower().replace(' ', '_')}_famd_pipeline.pkl"
    file_path = os.path.join(MODEL_SAVE_DIR, file_name)
    
    pipeline_data = {
        "model_name": m_name,
        "famd_transformers": famd_transformers,  
        "model": r["model_object"],  
        "target_encoder": le_target,
        "core_features": CORE_FEATURES
    }
    
    joblib.dump(pipeline_data, file_path)
    print(f"Saved FAMD pipeline '{m_name}' to '{file_path}'")



  SAVING ALL TRAINED FAMD PIPELINES
Saved FAMD pipeline 'Random Forest' to 'D:\2026\MockProject_062026_NhomAI\model\random_forest_famd_pipeline.pkl'
Saved FAMD pipeline 'Gradient Boosting' to 'D:\2026\MockProject_062026_NhomAI\model\gradient_boosting_famd_pipeline.pkl'
Saved FAMD pipeline 'XGBoost' to 'D:\2026\MockProject_062026_NhomAI\model\xgboost_famd_pipeline.pkl'
Saved FAMD pipeline 'LightGBM' to 'D:\2026\MockProject_062026_NhomAI\model\lightgbm_famd_pipeline.pkl'
